In [6]:
import pandas as pd
import numpy as np

from ta.momentum import RSIIndicator
from ta.trend import MACD
from ta.volatility import (
    BollingerBands,
    AverageTrueRange
)

# ==========================================
# LOAD CLEAN DATA
# ==========================================

df = pd.read_parquet(
    "../data/cleaned/btcusdt_1h_clean.parquet"
)

# ==========================================
# PRICE FEATURES
# ==========================================

df["return"] = df["close"].pct_change()

df["log_return"] = np.log(
    df["close"] / df["close"].shift(1)
)

df["high_low_spread"] = (
    df["high"] - df["low"]
) / df["close"]

df["open_close_spread"] = (
    df["close"] - df["open"]
) / df["open"]

# ==========================================
# TREND FEATURES
# ==========================================

df["ema20"] = (
    df["close"]
    .ewm(span=20)
    .mean()
)

df["ema50"] = (
    df["close"]
    .ewm(span=50)
    .mean()
)

df["ema100"] = (
    df["close"]
    .ewm(span=100)
    .mean()
)

df["ema200"] = (
    df["close"]
    .ewm(span=200)
    .mean()
)

df["ema20_ema50_ratio"] = (
    df["ema20"] / df["ema50"]
)

df["ema50_ema200_ratio"] = (
    df["ema50"] / df["ema200"]
)

# ==========================================
# MOMENTUM FEATURES
# ==========================================

rsi = RSIIndicator(
    close=df["close"],
    window=14
)

df["rsi"] = rsi.rsi()

macd = MACD(
    close=df["close"]
)

df["macd"] = macd.macd()

df["macd_signal"] = (
    macd.macd_signal()
)

df["macd_hist"] = (
    macd.macd_diff()
)

# ==========================================
# VOLATILITY FEATURES
# ==========================================

atr = AverageTrueRange(
    high=df["high"],
    low=df["low"],
    close=df["close"],
    window=14
)

df["atr"] = (
    atr.average_true_range()
)

df["volatility_24h"] = (
    df["return"]
    .rolling(24)
    .std()
)

df["volatility_168h"] = (
    df["return"]
    .rolling(168)
    .std()
)

# ==========================================
# VOLUME FEATURES
# ==========================================

df["volume_ma24"] = (
    df["volume"]
    .rolling(24)
    .mean()
)

df["volume_ratio"] = (
    df["volume"]
    / df["volume_ma24"]
)

df["volume_change"] = (
    df["volume"]
    .pct_change()
)

# ==========================================
# BOLLINGER BANDS
# ==========================================

bb = BollingerBands(
    close=df["close"],
    window=20,
    window_dev=2
)

df["bb_upper"] = (
    bb.bollinger_hband()
)

df["bb_lower"] = (
    bb.bollinger_lband()
)

df["bb_width"] = (
    (df["bb_upper"] - df["bb_lower"])
    / df["close"]
)

df["bb_position"] = (
    (df["close"] - df["bb_lower"])
    /
    (df["bb_upper"] - df["bb_lower"])
)

# ==========================================
# MARKET REGIME FEATURES
# ==========================================

df["price_vs_ema50"] = (
    df["close"]
    / df["ema50"]
)

df["price_vs_ema200"] = (
    df["close"]
    / df["ema200"]
)

# ==========================================
# CLEAN NaNs
# ==========================================

print("Rows before:", len(df))

df = (
    df
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
    .dropna()
    .reset_index(drop=True)
)

print("Rows after :", len(df))

# ==========================================
# FEATURE LIST
# ==========================================

feature_columns = [
    "return",
    "log_return",
    "high_low_spread",
    "open_close_spread",

    "ema20_ema50_ratio",
    "ema50_ema200_ratio",

    "rsi",

    "macd",
    "macd_signal",
    "macd_hist",

    "atr",

    "volatility_24h",
    "volatility_168h",

    "volume_ratio",
    "volume_change",

    "bb_width",
    "bb_position",

    "price_vs_ema50",
    "price_vs_ema200"
]

print("\nFeatures Generated:")
for feature in feature_columns:
    print(feature)

# ==========================================
# SAVE FEATURE DATASET
# ==========================================


# Create a copy for Excel export
export_df = df.copy()

# Excel cannot handle timezone-aware datetimes
export_df["open_time"] = (
    export_df["open_time"]
    .dt.tz_localize(None)
)

# Verify
print(export_df["open_time"].dtype)

export_df.to_parquet(
    "../data/features/btcusdt_features.parquet",
    index=False
)

# 2. Save to an Excel format that LibreOffice Calc opens
# Change 'Sheet1' to your preferred tab name
export_df.to_excel('../data/features/PPO.xlsx', index=False, sheet_name='Features')


print(
    "\nSaved -> btcusdt_features.parquet"
)

Rows before: 29927
Rows after : 29758

Features Generated:
return
log_return
high_low_spread
open_close_spread
ema20_ema50_ratio
ema50_ema200_ratio
rsi
macd
macd_signal
macd_hist
atr
volatility_24h
volatility_168h
volume_ratio
volume_change
bb_width
bb_position
price_vs_ema50
price_vs_ema200
datetime64[ns]

Saved -> btcusdt_features.parquet


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.9/250.9 kB 953.4 kB/s eta 0:00:0031m1.2 MB/s eta 0:00:01
Note: you may need to restart the kernel to use updated packages.
